# 🎙️ Real-Time Voice AI Backend – SniperThink Task

This notebook:
1. Installs dependencies
2. Writes all backend source files to disk
3. Sets your OpenAI API key
4. Starts the WebSocket server
5. Creates a public URL via **ngrok** so your browser frontend can connect

**Run cells top-to-bottom, one by one.**

## Step 1 – Install Dependencies

In [ ]:
%%capture
!pip install websockets openai numpy python-dotenv sentence-transformers faiss-cpu pyngrok --quiet

## Step 2 – Write source files to Colab disk

In [ ]:
import os
os.makedirs('/content/backend', exist_ok=True)
print('✅ Directory ready')

In [ ]:
%%writefile /content/backend/session.py
"""Session Manager – tracks active voice sessions."""
import logging
from typing import Dict
logger = logging.getLogger(__name__)

class SessionManager:
    def __init__(self):
        self._sessions: Dict[str, object] = {}
    def add(self, session_id, pipeline):
        self._sessions[session_id] = pipeline
        logger.info(f'Session added: {session_id}. Total: {len(self._sessions)}')
    def remove(self, session_id):
        self._sessions.pop(session_id, None)
        logger.info(f'Session removed: {session_id}. Total: {len(self._sessions)}')
    def get(self, session_id):
        return self._sessions.get(session_id)
    def count(self):
        return len(self._sessions)

In [ ]:
pipeline_code = '''
import asyncio, io, json, logging, os, struct, time
from typing import Optional
import numpy as np

logger = logging.getLogger(__name__)

SAMPLE_RATE = 16000
CHUNK_DURATION_MS = 20
SILENCE_THRESHOLD = 0.01
SILENCE_DURATION_MS = 700
VAD_WINDOW_CHUNKS = int(SILENCE_DURATION_MS / CHUNK_DURATION_MS)


class VoicePipeline:
    def __init__(self, session_id, websocket):
        self.session_id = session_id
        self.ws = websocket
        self._audio_buffer = b""
        self._silence_chunks = 0
        self._is_user_speaking = False
        self._speech_start_time = None
        self._state = "IDLE"
        self._processing_task = None
        self._tts_task = None
        self._conversation = [
            {"role": "system", "content": "You are a helpful, concise voice assistant. Keep responses SHORT (1-3 sentences). No markdown."}
        ]
        self._rag_retriever = None
        logger.info(f"[{self.session_id}] Pipeline initialized")

    async def handle_audio_chunk(self, chunk):
        rms = self._compute_rms(chunk)
        is_speech = rms > SILENCE_THRESHOLD
        if is_speech:
            self._silence_chunks = 0
            if not self._is_user_speaking:
                self._is_user_speaking = True
                self._speech_start_time = time.time()
                self._audio_buffer = b""
                await self._send_event("speech_started")
                if self._state == "SPEAKING":
                    await self.handle_interrupt()
            self._audio_buffer += chunk
        else:
            if self._is_user_speaking:
                self._silence_chunks += 1
                self._audio_buffer += chunk
                if self._silence_chunks >= VAD_WINDOW_CHUNKS:
                    await self.handle_end_of_speech()

    async def handle_end_of_speech(self):
        if not self._is_user_speaking and not self._audio_buffer:
            return
        self._is_user_speaking = False
        if len(self._audio_buffer) < 3200:
            self._audio_buffer = b""
            return
        audio_snapshot = self._audio_buffer
        self._audio_buffer = b""
        self._state = "PROCESSING"
        await self._send_event("processing_started")
        self._processing_task = asyncio.create_task(self._run_pipeline(audio_snapshot))

    async def handle_interrupt(self):
        logger.info(f"[{self.session_id}] Interrupt")
        if self._tts_task and not self._tts_task.done():
            self._tts_task.cancel()
        if self._processing_task and not self._processing_task.done():
            self._processing_task.cancel()
        self._state = "IDLE"
        await self._send_event("interrupted")

    async def cleanup(self):
        for t in [self._processing_task, self._tts_task]:
            if t and not t.done():
                t.cancel()

    async def _run_pipeline(self, audio):
        try:
            t0 = time.time()
            transcript = await self._transcribe(audio)
            if not transcript:
                self._state = "IDLE"; return
            logger.info(f"[{self.session_id}] STT ({(time.time()-t0)*1000:.0f}ms): {transcript!r}")
            await self._send_event("transcript", {"text": transcript})
            rag_context = ""
            if self._rag_retriever:
                rag_context = await self._retrieve_context(transcript)
            self._state = "SPEAKING"
            self._tts_task = asyncio.create_task(
                self._llm_and_tts(transcript, rag_context, (time.time()-t0)*1000)
            )
            await self._tts_task
        except asyncio.CancelledError:
            logger.info(f"[{self.session_id}] Pipeline cancelled")
        except Exception as e:
            logger.error(f"[{self.session_id}] Pipeline error: {e}", exc_info=True)
            await self._send_event("error", {"message": str(e)})
        finally:
            self._state = "IDLE"

    async def _transcribe(self, audio):
        api_key = os.getenv("OPENAI_API_KEY")
        if not api_key:
            return "[stub – set OPENAI_API_KEY]"
        try:
            import openai
            client = openai.AsyncOpenAI(api_key=api_key)
            wav_bytes = _pcm_to_wav(audio)
            f = io.BytesIO(wav_bytes); f.name = "audio.wav"
            response = await client.audio.transcriptions.create(model="whisper-1", file=f, response_format="text")
            return response.strip()
        except Exception as e:
            logger.error(f"STT error: {e}"); return ""

    async def _retrieve_context(self, query):
        try:
            docs = self._rag_retriever.retrieve(query, top_k=3)
            if docs:
                return "\n\nRelevant context:\n" + "\n\n".join(d["text"] for d in docs) + "\n"
        except Exception as e:
            logger.error(f"RAG error: {e}")
        return ""

    async def _llm_and_tts(self, user_text, rag_context, stt_latency_ms):
        api_key = os.getenv("OPENAI_API_KEY")
        if not api_key:
            await self._stream_tts_sentence("Please set your OpenAI API key."); return
        import openai, re
        client = openai.AsyncOpenAI(api_key=api_key)
        messages = list(self._conversation)
        content = (rag_context + "\nUser: " + user_text) if rag_context else user_text
        messages.append({"role": "user", "content": content})
        full_response = ""; sentence_buffer = ""
        try:
            stream = await client.chat.completions.create(
                model=os.getenv("LLM_MODEL", "gpt-4o-mini"),
                messages=messages, stream=True, max_tokens=150, temperature=0.7
            )
            async for chunk in stream:
                delta = chunk.choices[0].delta.content or ""
                full_response += delta; sentence_buffer += delta
                if any(c in sentence_buffer for c in [".", "!", "?", "\n"]):
                    parts = re.split(r"(?<=[.!?\n])\s*", sentence_buffer)
                    for s in parts[:-1]:
                        if s.strip():
                            await self._stream_tts_sentence(s.strip())
                    sentence_buffer = parts[-1] if parts else ""
            if sentence_buffer.strip():
                await self._stream_tts_sentence(sentence_buffer.strip())
        except asyncio.CancelledError:
            raise
        except Exception as e:
            logger.error(f"LLM error: {e}", exc_info=True)
        if full_response:
            self._conversation.append({"role": "user", "content": user_text})
            self._conversation.append({"role": "assistant", "content": full_response})
            if len(self._conversation) > 21:
                self._conversation = self._conversation[:1] + self._conversation[-20:]
        await self._send_event("response_complete", {"text": full_response})

    async def _stream_tts_sentence(self, text):
        api_key = os.getenv("OPENAI_API_KEY")
        if not api_key: return
        try:
            import openai
            client = openai.AsyncOpenAI(api_key=api_key)
            await self._send_event("tts_start", {"text": text})
            async with client.audio.speech.with_streaming_response.create(
                model="tts-1", voice=os.getenv("TTS_VOICE", "alloy"),
                input=text, response_format="pcm"
            ) as response:
                async for audio_chunk in response.iter_bytes(chunk_size=4096):
                    if asyncio.current_task().cancelled(): return
                    await self.ws.send(audio_chunk)
        except asyncio.CancelledError:
            raise
        except Exception as e:
            logger.error(f"TTS error: {e}")

    @staticmethod
    def _compute_rms(pcm_bytes):
        if len(pcm_bytes) < 2: return 0.0
        samples = np.frombuffer(pcm_bytes, dtype=np.int16).astype(np.float32)
        return float(np.sqrt(np.mean(samples ** 2)) / 32768.0)

    async def _send_event(self, event_type, data=None):
        try:
            payload = {"type": event_type}
            if data: payload.update(data)
            await self.ws.send(json.dumps(payload))
        except Exception:
            pass


def _pcm_to_wav(pcm_bytes, sample_rate=16000, channels=1):
    bits_per_sample = 16
    byte_rate = sample_rate * channels * bits_per_sample // 8
    block_align = channels * bits_per_sample // 8
    data_size = len(pcm_bytes)
    header = struct.pack(
        "<4sI4s4sIHHIIHH4sI",
        b"RIFF", 36 + data_size, b"WAVE",
        b"fmt ", 16, 1, channels,
        sample_rate, byte_rate, block_align, bits_per_sample,
        b"data", data_size,
    )
    return header + pcm_bytes
'''

with open('/content/backend/pipeline.py', 'w') as f:
    f.write(pipeline_code)
print('✅ pipeline.py written')

In [ ]:
%%writefile /content/backend/server.py
import asyncio, json, logging, os, uuid
import websockets
from pipeline import VoicePipeline
from session import SessionManager

logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s')
logger = logging.getLogger(__name__)
session_manager = SessionManager()

async def handle_client(websocket, path):
    session_id = str(uuid.uuid4())
    logger.info(f'New client: {session_id}')
    pipeline = VoicePipeline(session_id=session_id, websocket=websocket)
    session_manager.add(session_id, pipeline)
    try:
        await websocket.send(json.dumps({'type': 'session_started', 'session_id': session_id}))
        async for message in websocket:
            if isinstance(message, bytes):
                await pipeline.handle_audio_chunk(message)
            elif isinstance(message, str):
                data = json.loads(message)
                t = data.get('type')
                if t == 'interrupt': await pipeline.handle_interrupt()
                elif t == 'end_of_speech': await pipeline.handle_end_of_speech()
                elif t == 'ping': await websocket.send(json.dumps({'type': 'pong'}))
    except websockets.exceptions.ConnectionClosed:
        logger.info(f'Client disconnected: {session_id}')
    except Exception as e:
        logger.error(f'Error: {e}', exc_info=True)
    finally:
        await pipeline.cleanup()
        session_manager.remove(session_id)

async def main():
    host = os.getenv('HOST', '0.0.0.0')
    port = int(os.getenv('PORT', 8765))
    logger.info(f'Voice AI WebSocket server on ws://{host}:{port}')
    async with websockets.serve(handle_client, host, port,
                                ping_interval=20, ping_timeout=10,
                                max_size=1024*1024):
        await asyncio.Future()

if __name__ == '__main__':
    asyncio.run(main())

## Step 3 – Set your OpenAI API Key

In [ ]:
import os

# ✏️  PASTE YOUR KEY BELOW
os.environ['OPENAI_API_KEY'] = 'sk-YOUR-KEY-HERE'
os.environ['LLM_MODEL']      = 'gpt-4o-mini'   # or gpt-3.5-turbo
os.environ['TTS_VOICE']      = 'alloy'          # alloy, echo, nova, onyx, fable, shimmer
os.environ['PORT']           = '8765'

print('✅ Environment variables set')

## Step 4 – Start the WebSocket server (in background)

In [ ]:
import subprocess, sys, time

proc = subprocess.Popen(
    [sys.executable, '/content/backend/server.py'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    env={**os.environ},
    cwd='/content/backend'
)
time.sleep(2)  # give it a moment to start

if proc.poll() is None:
    print('✅ Server running (PID:', proc.pid, ')')
else:
    out, _ = proc.communicate()
    print('❌ Server failed to start:')
    print(out.decode())

## Step 5 – Expose with ngrok (public WebSocket URL)

Sign up free at https://ngrok.com → copy your authtoken from the dashboard.

In [ ]:
from pyngrok import ngrok, conf

# ✏️  PASTE YOUR NGROK AUTHTOKEN BELOW
NGROK_TOKEN = 'YOUR_NGROK_AUTHTOKEN_HERE'
ngrok.set_auth_token(NGROK_TOKEN)

tunnel = ngrok.connect(8765, 'tcp')
public_url = tunnel.public_url

# Convert tcp:// to wss://
ws_url = public_url.replace('tcp://', 'wss://')
print('\n' + '='*60)
print('🌐 Your public WebSocket URL:')
print(f'   {ws_url}')
print('='*60)
print('\nPaste this URL into the frontend input box.')

## Step 6 – Optionally enable RAG

In [ ]:
# Run this cell only if you want to test RAG
import sys
sys.path.insert(0, '/content/backend')

# Write a small knowledge base text file
kb_text = """
SniperThink is an AI-first company building intelligent automation tools.
Our voice AI system supports real-time bidirectional audio at sub-second latency.
The system uses Whisper for STT, GPT-4o-mini for LLM, and OpenAI TTS for speech output.
RAG is implemented using FAISS vector search with sentence-transformers embeddings.
The WebSocket-based architecture ensures minimal overhead for streaming audio.
"""

with open('/content/knowledge_base.txt', 'w') as f:
    f.write(kb_text)

print('✅ Knowledge base saved to /content/knowledge_base.txt')
print('RAG is loaded automatically when the pipeline starts.')

## Step 7 – View Server Logs

Run this cell any time to see the latest server output.

In [ ]:
import subprocess
# Tail last 30 lines of server output
try:
    out = proc.stdout.read1(4096).decode(errors='replace')
    print(out if out else '(no new output)')
except Exception as e:
    print(f'Log read error: {e}')

## Step 8 – Stop the server (when done)

In [ ]:
ngrok.disconnect(tunnel.public_url)
proc.terminate()
print('✅ Server stopped')